## Gold Layer - Revenue Dashboard

In [0]:
%run ./00_config

In [0]:
%sql
CREATE TABLE IF NOT EXISTS nyc_taxi_project.gold.revenue_dashboard
(
  window_start          TIMESTAMP,
  window_end            TIMESTAMP,
  pickup_borough        STRING,
  payment_type_desc     STRING,
  total_trips           LONG,
  total_revenue         DOUBLE,
  total_tips            DOUBLE,
  avg_fare              DOUBLE,
  avg_tip_pct           DOUBLE,
  avg_trip_distance     DOUBLE,
  avg_duration_minutes  DOUBLE,
  airport_trips         LONG,
  rush_hour_trips       LONG
)
USING DELTA
COMMENT 'Gold 1 - Revenue by 1-hour tumbling window + watermark';

In [0]:
from pyspark.sql import functions as F

def process_gold_revenue():

    query = (spark.readStream
                .format("delta")
                .table(SILVER2_TABLE)

             .withWatermark("tpep_pickup_datetime", "1 day")

             .groupBy(
                 F.window("tpep_pickup_datetime", "1 hour"),
                 "pickup_borough",
                 "payment_type_desc"
             )
             .agg(
                 F.count("*")                              .alias("total_trips"),
                 F.round(F.sum("total_amount"), 2)         .alias("total_revenue"),
                 F.round(F.sum("tip_amount"), 2)           .alias("total_tips"),
                 F.round(F.avg("fare_amount"), 2)          .alias("avg_fare"),
                 F.round(F.avg("tip_percentage"), 2)       .alias("avg_tip_pct"),
                 F.round(F.avg("trip_distance"), 2)        .alias("avg_trip_distance"),
                 F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
                 F.sum(F.col("is_airport_trip").cast("int")).alias("airport_trips"),
                 F.sum(F.col("is_rush_hour").cast("int"))  .alias("rush_hour_trips")
             )
             .withColumn("window_start", F.col("window.start"))
             .withColumn("window_end",   F.col("window.end"))
             .drop("window")

             .writeStream
                .format("delta")
                .outputMode("append")
                .option("checkpointLocation", GOLD_REV_CHECKPOINT)
                .trigger(availableNow=True)
                .toTable(GOLD_REVENUE))

    query.awaitTermination()
    
process_gold_revenue()

In [0]:
%sql
SELECT
    DATE_TRUNC('month', window_start)   AS month,
    pickup_borough,
    payment_type_desc,
    SUM(total_trips)                    AS total_trips,
    ROUND(SUM(total_revenue), 2)        AS total_revenue_usd,
    ROUND(AVG(avg_fare), 2)             AS avg_fare,
    ROUND(AVG(avg_tip_pct), 2)          AS avg_tip_pct
FROM nyc_taxi_project.gold.revenue_dashboard
GROUP BY 1, 2, 3
ORDER BY 1, total_revenue_usd DESC
LIMIT 10;